[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmusachio/loans-credit-policies/blob/main/loans_credit_policies.ipynb)

# **Políticas de Crédito e Precificação Baseada em Risco**

O **Lending Club** é uma plataforma de empréstimos *peer-to-peer* (P2P) fundada em 2007 nos Estados Unidos, conectando diretamente tomadores de crédito a investidores individuais e institucionais. Os contratos emitidos destinam-se a finalidades diversas, como consolidação de dívidas, reformas residenciais e financiamento de pequenos negócios.

Na gestão de risco de crédito, prever apenas um rótulo binário (adimplente vs. inadimplente) é insuficiente para a tomada de decisão financeira. A concessão sustentável depende da estimação precisa e **calibrada** da **Probabilidade de Default ($\text{PD}$)** no momento da solicitação do crédito, permitindo quantificar a **Perda Esperada ($\text{Expected Loss}$)** de cada operação:

>$$\text{Losses}(x) = \text{PD}(x) \times \text{EAD}(x) \times \text{LGD}(x)$$

>* **$\text{PD}(x)$ (*Probability of Default*):** Probabilidade de inadimplência estimada pelo modelo de Machine Learning com variáveis restritas ao momento da aplicação.
>* **$\text{EAD}(x)$ (*Exposure at Default*):** Parcela do montante total devido (principal + juros contratados) que permanece em aberto no momento do *default*.
>* **$\text{LGD}(x)$ (*Loss Given Default*):** Severidade da perda efetiva após deduzidas eventuais recuperações de cobrança ($\text{recoveries}$).

## Setup
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import seaborn as sns
import shap

from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve

plt.rcParams.update({"figure.dpi": 150})
custom_colors = ["#9b59b6", "#3498db", "#95a5a6", "#e74c3c", "#34495e", "#2ecc71"]
sns.set_theme(style="whitegrid", font_scale=1, palette=custom_colors)

In [ ]:
def clf_metric_report(y_true, y_score, model_name="Model"):
    return pd.DataFrame(
        {
            "model": [model_name],
            "log_loss": [log_loss(y_true, y_score)],
            "brier_score": [brier_score_loss(y_true, y_score)],
            "roc_auc": [roc_auc_score(y_true, y_score)],
            "average_precision": [average_precision_score(y_true, y_score)],
        }
    ).set_index("model")


def compute_and_plot_permutation_importance(
    model, X_eval, y_eval, metric="average_precision", n_repeats=5, top_n=20
):
    result = permutation_importance(
        model, X_eval, y_eval, n_repeats=n_repeats, random_state=42, scoring=metric
    )
    feature_importance = (
        pd.DataFrame({"feature": X_eval.columns, "importance": result.importances_mean})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    plt.figure(figsize=(10, 8))
    sns.barplot(data=feature_importance.head(top_n), y="feature", x="importance")
    plt.xlabel(f"Importância por Permutação ({metric})")
    plt.ylabel("Variáveis")
    plt.title(f"Top {top_n} Variáveis Mais Importantes")
    plt.tight_layout()
    plt.show()

    return feature_importance


def plot_calibration_curve(y_true, y_score, n_bins=10, title="Curva de Calibração"):
    prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=n_bins)
    plt.figure(figsize=(6, 6))
    plt.plot(prob_pred, prob_true, marker="o", label="Modelo")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfeitamente Calibrado")
    plt.xlabel("Probabilidade Predita (PD)")
    plt.ylabel("Frequência Observada de Default")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_pr_calib_curve(y_true, y_score, n_bins=10):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=n_bins)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(recall, precision, marker=".")
    axes[0].set_xlabel("Recall")
    axes[0].set_ylabel("Precision")
    axes[0].set_title("Curva Precision-Recall")

    axes[1].plot(prob_pred, prob_true, marker="o", label="Modelo")
    axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Ideal")
    axes[1].set_xlabel("Probabilidade Predita (PD)")
    axes[1].set_ylabel("Frequência Observada de Default")
    axes[1].set_title("Curva de Calibração")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_dis_probs(y_true, y_score):
    plt.figure(figsize=(10, 6))
    sns.histplot(
        y_score[y_true == 1],
        bins=50,
        color="#e74c3c",
        label="Inadimplente (Default = 1)",
        kde=True,
        stat="density",
    )
    sns.histplot(
        y_score[y_true == 0],
        bins=50,
        color="#3498db",
        label="Adimplente (Default = 0)",
        kde=True,
        stat="density",
    )
    plt.xlabel("Probabilidade Predita de Default (PD)")
    plt.ylabel("Densidade")
    plt.title("Distribuição das Probabilidades Preditas: Adimplentes vs. Inadimplentes")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_shap_values(shap_values, X_eval, max_display=25):
    shap.summary_plot(shap_values, X_eval, plot_type="bar", max_display=max_display)

## Sample
---

### load data + target definition

In [ ]:
DATA_URL = "hf://datasets/pmusachio/loan_status_2007-2020Q3/Loan_status_2007-2020Q3.parquet"

df_raw = pd.read_parquet(DATA_URL)
df_raw.loc[:, "default"] = df_raw["loan_status"].map({"Fully Paid": 0, "Charged Off": 1})
df_raw.head()

### out-of-time split (2007–2017 vs 2018)
* `df_train`: issue_d < '2018-01-01'
* `df_test`: '2018-01-01' <= issue_d <= '2019-01-01'

In [ ]:
df_train = df_raw[df_raw["issue_d"] < "2018-01-01"].copy()
df_test = df_raw[(df_raw["issue_d"] >= "2018-01-01") & (df_raw["issue_d"] <= "2019-01-01")].copy()

pd.DataFrame(
    {
        "partition": ["Train (2007–2017)", "Test OOT (2018)"],
        "total_rows": [len(df_train), len(df_test)],
        "resolved_loans": [df_train["default"].notna().sum(), df_test["default"].notna().sum()],
        "default_rate": [df_train["default"].mean(), df_test["default"].mean()],
    }
).set_index("partition")

## Explore
---

### volume de emprestimos no tempo

In [ ]:
df_loans_over_time = df_raw.groupby("issue_d", observed=True)["id"].count().reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=df_loans_over_time, x="issue_d", y="id", ax=ax)
ax.set_xlabel("Data de Emissão (issue_d)")
ax.set_ylabel("Total de Contratos")
ax.set_title("Volume Total de Empréstimos Emitidos ao Longo do Tempo")
plt.tight_layout()
plt.show()

df_loans_over_time.tail()

aumento no volume de contratos a partir de 2012 reflete a expansão da operação nos Estados Unidos, seguido por oscilações cíclicas que reforçam a necessidade de validar o modelo em safras temporais distintas

### evolucao da taxa de inadimplencia (default ratio)

In [ ]:
df_default_over_time = df_raw.groupby("issue_d", observed=True)["default"].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=df_default_over_time, x="issue_d", y="default", ax=ax)
ax.set_xlabel("Data de Emissão (issue_d)")
ax.set_ylabel("Taxa de Default")
ax.set_title("Inadimplência ao Longo do Tempo")
plt.tight_layout()
plt.show()

df_default_over_time.dropna().tail()

### comparativo temporal --> treino (2007–2017) vs teste (2018)

comparando lado a lado a curva histórica de inadimplência do período de desenvolvimento (`2007–2017`) e a janela de avaliação (`2018`) procurando mudanças de regime na carteira

In [ ]:
df_gp_train = df_train.groupby("issue_d", observed=True)["default"].mean().reset_index()
df_gp_test = df_test.groupby("issue_d", observed=True)["default"].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=df_gp_train, x="issue_d", y="default", label="Treino (2007–2017)", ax=ax)
sns.lineplot(data=df_gp_test, x="issue_d", y="default", label="Teste OOT (2018)", ax=ax)
ax.set_xlabel("Data de Emissão (issue_d)")
ax.set_ylabel("Taxa de Default")
ax.set_title("Taxa de Inadimplência: Treino (2007–2017) vs. Teste Out-of-Time (2018)")
ax.legend()
plt.tight_layout()
plt.show()

## Modify
---

1. **Exclusão de Variáveis Pós-Concessão (*Anti-Leakage*):** Remoção de todas as colunas alimentadas após a emissão do contrato (ex.: histórico de pagamentos realizados `total_pymnt`, recuperações `recoveries`, saldo devedor `out_prncp` e atualizações futuras de bureau `last_fico_range_high`). Essas variáveis contábeis permanecem reservadas apenas para a aferição financeira de $\text{EAD}$ e $\text{LGD}$ na etapa **Assess**.
2. **Engenharia de Atributos (*Feature Engineering*):** Criação de razões financeiras e indicadores de comprometimento de renda no momento da solicitação.
3. **Seleção de Variáveis (*Feature Selection*):** Filtragem para um conjunto enxuto, estável e auditável de preditores.

## Model
---

1. **Estratégia de Validação Interna:** Utilização exclusiva da janela `2007–2017` para treino, validação (*tuning* de hiperparâmetros) e calibração *out-of-sample*.
2. **Métrica de Otimização (Log Loss):** Como as probabilidades preditas alimentam diretamente a equação financeira de perda esperada, o modelo é treinado e calibrado visando minimizar o **Log Loss** e o **Brier Score**.

## Assess
---

### benchmarks de referencia

### formulacao da base de calibração ($\text{EAD}$, $\text{LGD}$ e $\text{Losses}$)

| indicador | lógica |
| --- | :--- |
| **Montante Total Devido ($\text{Principal}$):** | $$\text{Principal} = \text{loan\_amnt} \times (1 + \text{int\_rate})$$ |
| **Exposição no Default ($\text{EAD}$ e $\text{EAD Ratio}$):** | $$\text{EAD} = \text{Principal} - \text{total\_pymnt}, \qquad \text{EAD Ratio} = \frac{\text{EAD}}{\text{Principal}}$$ |
| **Severidade da Perda Dado o Default ($\text{LGD Ratio}$):** | $$\text{LGD Ratio} = 1 - \frac{\text{recoveries}}{\text{EAD}}$$ |
| **Perda Financeira Realizada ($\text{Losses}$):** | $$\text{Losses} = \text{Principal} - \text{total\_pymnt} + \text{recoveries}$$ |

In [ ]:
def compute_credit_loss_components(df):
    result = df.copy()
    result["principal"] = result["loan_amnt"] * (1 + result["int_rate"])
    result["ead"] = result["principal"] - result["total_pymnt"]
    result["ead_ratio"] = result["ead"] / result["principal"]
    result["lgd_ratio"] = 1 - (result["recoveries"] / result["ead"])
    result["losses"] = result["principal"] - result["total_pymnt"] + result["recoveries"]
    return result


def evaluate_predictions_log_loss(test_df_with_target, predictions_df):
    merged = test_df_with_target[["id", "default"]].merge(
        predictions_df[["id", "probabilities"]], on="id"
    ).dropna()
    return pd.DataFrame(
        {
            "evaluated_loans": [len(merged)],
            "log_loss": [log_loss(merged["default"], merged["probabilities"])],
        }
    )